In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# Plot style
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "font.size":        11,
})
 
print("Libraries loaded.")

In [ ]:
df = pd.read_csv("lgbm_second_run.clustermap_clusters.csv")
df.head()

In [ ]:
# Parse dates
df["start_date"] = pd.to_datetime(df["start_date"], format="%Y/%m/%d")
df["end_date"]   = pd.to_datetime(df["end_date"],   format="%Y/%m/%d")
 
# Midpoint date — useful for placing a cluster on a timeline
df["mid_date"] = df["start_date"] + (df["end_date"] - df["start_date"]) / 2
 
# Numeric casts (should already be numeric, but belt-and-suspenders)
num_cols = ["observed_cases", "expected_cases", "excess_cases",
            "obs_exp_ratio", "llr", "radius_km", "duration_days",
            "duration_weeks", "center_lat", "center_lng",
            "n_case_points", "center_to_case_centroid_km"]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
 
# Boolean casts
df["is_high_rate"]    = df["is_high_rate"].astype(str).str.lower() == "true"
df["is_hierarchical"] = df["is_hierarchical"].astype(str).str.lower() == "true"
 
print(f"Loaded {len(df)} clusters.")
print(f"Date range: {df['start_date'].min().date()} → {df['end_date'].max().date()}")
df.head()

In [ ]:
summary_cols = ["observed_cases", "expected_cases", "excess_cases",
                "obs_exp_ratio", "llr", "radius_km",
                "duration_days", "n_case_points"]
 
print("=== Descriptive Statistics ===\n")
display(df[summary_cols].describe().round(2))
 
print("\n=== Cluster Flag Counts ===")
print(f"  High-rate clusters:    {df['is_high_rate'].sum()} / {len(df)}")
print(f"  Hierarchical clusters: {df['is_hierarchical'].sum()} / {len(df)}")
 
print("\n=== Temporal Coverage ===")
print(f"  Earliest start:  {df['start_date'].min().date()}")
print(f"  Latest end:      {df['end_date'].max().date()}")
print(f"  Median duration: {df['duration_days'].median():.0f} days  "
      f"({df['duration_weeks'].median():.1f} weeks)")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Distribution of Key Cluster Statistics", fontsize=14, fontweight="bold", y=1.01)
 
panels = [
    ("llr",            "Log-Likelihood Ratio (LLR)",  "steelblue"),
    ("obs_exp_ratio",  "Observed / Expected Ratio",   "darkorange"),
    ("observed_cases", "Observed Cases",               "mediumseagreen"),
    ("excess_cases",   "Excess Cases (Obs - Exp)",    "crimson"),
    ("duration_days",  "Cluster Duration (days)",      "mediumpurple"),
    ("radius_km",      "Cluster Radius (km)",          "saddlebrown"),
]
 
for ax, (col, title, color) in zip(axes.flat, panels):
    data = df[col].dropna()
    ax.hist(data, bins=40, color=color, alpha=0.8, edgecolor="white", linewidth=0.4)
    ax.axvline(data.median(), color="black", linestyle="--", linewidth=1.2,
               label=f"Median: {data.median():.1f}")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)
 
plt.tight_layout()
plt.show()

In [ ]:
df["start_month_period"] = df["start_date"].dt.to_period("M")
monthly_starts = df.groupby("start_month_period").size().reset_index(name="n_clusters")

# Reindex to full continuous range so every month appears on the axis
full_range = pd.period_range(
    start=df["start_date"].min().to_period("M"),
    end=df["end_date"].max().to_period("M"),
    freq="M"
)
monthly_starts = (
    monthly_starts
    .set_index("start_month_period")
    .reindex(full_range, fill_value=0)
    .reset_index()
    .rename(columns={"index": "start_month_period"})
)
monthly_starts["date"] = monthly_starts["start_month_period"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(monthly_starts["date"], monthly_starts["n_clusters"],
       width=20, color="steelblue", alpha=0.85, edgecolor="white")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.xticks(rotation=45, ha="right")
ax.set_title("Number of Clusters Starting Each Month", fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Number of Clusters")
plt.tight_layout()
plt.show()

In [ ]:
monthly_cases = df.groupby("start_month_period").agg(
    total_observed = ("observed_cases", "sum"),
    total_excess   = ("excess_cases",   "sum"),
    mean_obs_exp   = ("obs_exp_ratio",  "mean"),
).reset_index()
monthly_cases["date"] = monthly_cases["start_month_period"].dt.to_timestamp()
 
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
fig.suptitle("Monthly Cluster Burden (by Cluster Start Month)",
             fontsize=13, fontweight="bold")
 
# Top: observed vs excess
axes[0].bar(monthly_cases["date"], monthly_cases["total_observed"],
            width=20, label="Total Observed Cases", color="steelblue", alpha=0.8)
axes[0].bar(monthly_cases["date"], monthly_cases["total_excess"],
            width=20, label="Excess Cases",         color="crimson",   alpha=0.6)
axes[0].set_ylabel("Case Count")
axes[0].legend()
 
# Bottom: mean obs/exp ratio
axes[1].plot(monthly_cases["date"], monthly_cases["mean_obs_exp"],
             color="darkorange", linewidth=2, marker="o", markersize=5)
axes[1].axhline(1, color="gray", linestyle="--", linewidth=1, label="Expected = Observed")
axes[1].set_ylabel("Mean Obs/Exp Ratio")
axes[1].set_xlabel("Month")
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.xticks(rotation=45, ha="right")
 
plt.tight_layout()
plt.show()

In [ ]:
top_n = 60
top_df = df.nlargest(top_n, "llr").sort_values("start_date")
 
norm = mcolors.Normalize(vmin=top_df["llr"].min(), vmax=top_df["llr"].max())
cmap = cm.get_cmap("YlOrRd")
 
fig, ax = plt.subplots(figsize=(15, 12))
 
for i, (_, row) in enumerate(top_df.iterrows()):
    color = cmap(norm(row["llr"]))
    ax.barh(
        y=i,
        left=mdates.date2num(row["start_date"]),
        width=mdates.date2num(row["end_date"]) - mdates.date2num(row["start_date"]),
        color=color, edgecolor="white", linewidth=0.3, height=0.7
    )
 
ax.set_yticks(range(len(top_df)))
ax.set_yticklabels(
    [f"C{r['cluster_id']}  ({r['duration_days']}d)" for _, r in top_df.iterrows()],
    fontsize=7.5
)
ax.xaxis_date()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax.set_title(f"Top {top_n} Clusters by LLR — Duration Timeline (Gantt)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date")
 
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label="Log-Likelihood Ratio (LLR)", shrink=0.5)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
 
scatter = ax.scatter(
    df["obs_exp_ratio"],
    df["llr"],
    c=df["duration_days"],
    s=df["observed_cases"] / df["observed_cases"].max() * 400 + 10,
    cmap="plasma",
    alpha=0.7,
    edgecolors="white",
    linewidths=0.4
)
 
plt.colorbar(scatter, label="Duration (days)")
 
# Label top 10 by LLR
for _, row in df.nlargest(10, "llr").iterrows():
    ax.annotate(
        f"C{int(row['cluster_id'])}",
        (row["obs_exp_ratio"], row["llr"]),
        textcoords="offset points", xytext=(6, 3),
        fontsize=8, color="black"
    )
 
ax.set_xlabel("Observed / Expected Ratio", fontsize=12)
ax.set_ylabel("Log-Likelihood Ratio (LLR)", fontsize=12)
ax.set_title("LLR vs Obs/Exp Ratio\n(bubble size = observed cases, color = duration)",
             fontsize=13, fontweight="bold")
 
# Legend for bubble size
for cases, label in [(50, "50 cases"), (500, "500 cases"), (2000, "2000 cases")]:
    ax.scatter([], [], s=cases / df["observed_cases"].max() * 400 + 10,
               c="gray", alpha=0.6, label=label)
ax.legend(title="Observed Cases", framealpha=0.7, fontsize=9)
 
plt.tight_layout()
plt.show()

# Analyzing Large Area Clusters

In [ ]:
# ── Clusters to investigate ───────────────────────────────────────────────────
TARGET_IDS = [28, 151, 307, 308]
 
targets = df[df["cluster_id"].isin(TARGET_IDS)].copy()
print("Target clusters loaded:\n")
display(
    targets[[
        "cluster_id", "radius_km", "duration_days",
        "observed_cases", "expected_cases", "obs_exp_ratio", "llr",
        "center_lat", "center_lng",
        "cases_centroid_lat", "cases_centroid_lng",
        "center_to_case_centroid_km",
        "start_date", "end_date"
    ]].set_index("cluster_id").round(3)
)

In [ ]:
# Key diagnostic: offset between cluster center and case centroid
# =============================================================================
 
print("=== Cluster Center vs Case Centroid Offset ===\n")
for _, row in targets.iterrows():
    offset  = row["center_to_case_centroid_km"]
    radius  = row["radius_km"]
    pct     = (offset / radius) * 100
    print(f"  Cluster {int(row['cluster_id'])}:")
    print(f"    Cluster radius:               {radius:.2f} km")
    print(f"    Center → case centroid offset: {offset:.2f} km")
    print(f"    Offset as % of radius:         {pct:.1f}%")
    print(f"    Interpretation: The center of mass of the cases sits")
    print(f"    {pct:.0f}% of the way from the cluster center to its edge.")
    print()

In [ ]:
# Side-by-side spatial map of each cluster
#           Shows: cluster circle, declared center, case centroid, case bbox
# =============================================================================
 
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle(
    "Cluster Circle vs Case Distribution\n"
    "(★ = declared center, ✦ = case centroid, dashed box = case bounding box)",
    fontsize=13, fontweight="bold"
)
 
colors = ["steelblue", "darkorange", "mediumseagreen", "crimson"]
 
for ax, (_, row), color in zip(axes.flat, targets.iterrows(), colors):
    clat = row["center_lat"]
    clng = row["center_lng"]
    r_km = row["radius_km"]
 
    # Convert radius from km to approximate degrees for plotting
    # 1 deg lat ≈ 111 km;  1 deg lng ≈ 111 * cos(lat) km
    r_lat_deg = r_km / 111.0
    r_lng_deg = r_km / (111.0 * np.cos(np.radians(clat)))
 
    # Draw cluster circle (approximate ellipse in lat/lng space)
    ellipse = mpatches.Ellipse(
        (clng, clat),
        width=2 * r_lng_deg,
        height=2 * r_lat_deg,
        fill=True, facecolor=color, alpha=0.15,
        edgecolor=color, linewidth=2, linestyle="--"
    )
    ax.add_patch(ellipse)
 
    # Declared center
    ax.scatter(clng, clat, marker="*", s=300, color=color,
               zorder=5, label="Declared center", edgecolors="white", linewidths=0.5)
 
    # Case centroid
    ax.scatter(row["cases_centroid_lng"], row["cases_centroid_lat"],
               marker="D", s=120, color="black",
               zorder=5, label="Case centroid", edgecolors="white", linewidths=0.5)
 
    # Line connecting center to case centroid
    ax.plot(
        [clng, row["cases_centroid_lng"]],
        [clat, row["cases_centroid_lat"]],
        color="black", linewidth=1.5, linestyle=":", zorder=4,
        label=f"Offset: {row['center_to_case_centroid_km']:.2f} km"
    )
 
    # Case bounding box
    bbox_lngs = [row["cases_bbox_lng_min"], row["cases_bbox_lng_max"],
                 row["cases_bbox_lng_max"], row["cases_bbox_lng_min"],
                 row["cases_bbox_lng_min"]]
    bbox_lats = [row["cases_bbox_lat_min"], row["cases_bbox_lat_min"],
                 row["cases_bbox_lat_max"], row["cases_bbox_lat_max"],
                 row["cases_bbox_lat_min"]]
    ax.plot(bbox_lngs, bbox_lats, color="gray", linewidth=1,
            linestyle="--", alpha=0.7, label="Case bounding box")
 
    # Formatting
    pad_lng = r_lng_deg * 1.25
    pad_lat = r_lat_deg * 1.25
    ax.set_xlim(clng - pad_lng, clng + pad_lng)
    ax.set_ylim(clat - pad_lat, clat + pad_lat)
    ax.set_title(
        f"Cluster {int(row['cluster_id'])}  |  r = {r_km:.1f} km  |  "
        f"offset = {row['center_to_case_centroid_km']:.1f} km  "
        f"({row['center_to_case_centroid_km']/r_km*100:.0f}% of radius)\n"
        f"{row['start_date'].strftime('%Y-%m-%d')} → {row['end_date'].strftime('%Y-%m-%d')}  "
        f"|  obs={int(row['observed_cases'])}  obs/exp={row['obs_exp_ratio']:.2f}x  "
        f"LLR={row['llr']:.1f}",
        fontsize=9, fontweight="bold"
    )
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.legend(fontsize=8, loc="upper right")
 
plt.tight_layout()
plt.show()

In [ ]:
# CELL D — What fraction of the circle area actually contains cases?
#           Estimate using the bounding box of cases vs circle area
# =============================================================================
 
print("=== Case Coverage Analysis ===\n")
print(f"{'Cluster':<10} {'Circle Area':>14} {'BBox Area':>12} {'Coverage %':>12} {'Offset % radius':>16}")
print("-" * 68)
 
for _, row in targets.iterrows():
    circle_area = np.pi * row["radius_km"] ** 2
    bbox_area   = row["cases_bbox_lat_span_km"] * row["cases_bbox_lng_span_km"]
    coverage    = (bbox_area / circle_area) * 100
    offset_pct  = (row["center_to_case_centroid_km"] / row["radius_km"]) * 100
 
    print(
        f"  C{int(row['cluster_id']):<7} "
        f"{circle_area:>12.1f} km²  "
        f"{bbox_area:>10.1f} km²  "
        f"{coverage:>10.1f}%  "
        f"{offset_pct:>14.1f}%"
    )
 
print("""
Interpretation:
  'Coverage %' is the bounding box of actual cases as a fraction of the full
  circle area. A low number means cases are packed into a small corner of
  the overall cluster polygon — exactly what you're seeing on the map.
""")

In [ ]:
# CELL E — Why does SaTScan do this? The LLR surface explanation
#
# SaTScan maximizes the LLR by trying many candidate circles. A large circle
# centered slightly away from the actual cases can still have a high LLR if:
#   1. The expected count inside the large circle is low (sparse background)
#   2. All the real cases happen to fall within that circle
#   3. The edge-hugging position concentrates the signal at the boundary
#
# We can visualize this by showing obs/exp vs radius for the target clusters
# compared to the full dataset.
# =============================================================================
 
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Target Clusters in Context: Radius & Offset vs LLR",
             fontsize=13, fontweight="bold")
 
# Left: radius vs LLR, highlight targets
ax = axes[0]
ax.scatter(df["radius_km"], df["llr"],
           s=20, alpha=0.3, color="lightgray", label="All clusters")
for (_, row), color in zip(targets.iterrows(), colors):
    ax.scatter(row["radius_km"], row["llr"],
               s=200, color=color, zorder=5, edgecolors="white", linewidths=1,
               label=f"C{int(row['cluster_id'])}  (offset {row['center_to_case_centroid_km']:.1f}km)")
    ax.annotate(
        f"  C{int(row['cluster_id'])}",
        (row["radius_km"], row["llr"]),
        fontsize=9, color=color, fontweight="bold"
    )
ax.set_xlabel("Cluster Radius (km)")
ax.set_ylabel("LLR")
ax.set_title("Radius vs LLR\n(large radius + moderate LLR = edge-hugging candidate)")
ax.legend(fontsize=8)
 
# Right: center-to-centroid offset vs obs/exp ratio, all clusters
ax = axes[1]
sc = ax.scatter(
    df["center_to_case_centroid_km"],
    df["obs_exp_ratio"],
    c=df["radius_km"],
    cmap="YlOrRd", s=25, alpha=0.5, edgecolors="none"
)
plt.colorbar(sc, ax=ax, label="Radius (km)")
for (_, row), color in zip(targets.iterrows(), colors):
    ax.scatter(
        row["center_to_case_centroid_km"], row["obs_exp_ratio"],
        s=200, color=color, zorder=5, edgecolors="white", linewidths=1,
        label=f"C{int(row['cluster_id'])}"
    )
    ax.annotate(
        f"  C{int(row['cluster_id'])}",
        (row["center_to_case_centroid_km"], row["obs_exp_ratio"]),
        fontsize=9, color=color, fontweight="bold"
    )
ax.set_xlabel("Center → Case Centroid Offset (km)")
ax.set_ylabel("Obs/Exp Ratio")
ax.set_title("Offset vs Obs/Exp Ratio\n(high offset + moderate obs/exp = edge-hugging cluster)")
ax.legend(fontsize=8)
 
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELL F — Diagnostic Summary (continuous metrics, no arbitrary thresholds)
# =============================================================================

print("=== Diagnostic Summary ===\n")

# Compute dataset-wide percentiles for context
radius_pct_rank    = lambda r: (df["radius_km"] < r).mean() * 100
offset_pct_rank    = lambda o: (df["center_to_case_centroid_km"] < o).mean() * 100
llr_pct_rank       = lambda l: (df["llr"] < l).mean() * 100
obs_exp_pct_rank   = lambda o: (df["obs_exp_ratio"] < o).mean() * 100

for _, row in targets.iterrows():
    radius      = row["radius_km"]
    offset      = row["center_to_case_centroid_km"]
    offset_pct  = offset / radius * 100
    circle_area = np.pi * radius ** 2
    bbox_area   = row["cases_bbox_lat_span_km"] * row["cases_bbox_lng_span_km"]
    coverage    = bbox_area / circle_area * 100

    print(f"Cluster {int(row['cluster_id'])}:")
    print(f"  ── Spatial geometry ──────────────────────────────────────────")
    print(f"  Cluster radius:                {radius:.2f} km  "
          f"({radius_pct_rank(radius):.0f}th percentile of all clusters)")
    print(f"  Center → case centroid offset: {offset:.2f} km  "
          f"({offset_pct:.1f}% of radius)  "
          f"[{offset_pct_rank(offset):.0f}th percentile of all clusters]")
    print(f"  Case bbox area:                {bbox_area:.1f} km²")
    print(f"  Circle area:                   {circle_area:.1f} km²")
    print(f"  Case bbox as % of circle:      {coverage:.1f}%")
    print()
    print(f"  ── Statistical signal ────────────────────────────────────────")
    print(f"  Observed cases:                {int(row['observed_cases'])}")
    print(f"  Expected cases:                {row['expected_cases']:.2f}")
    print(f"  Excess cases:                  {row['excess_cases']:.2f}")
    print(f"  Obs/Exp ratio:                 {row['obs_exp_ratio']:.2f}x  "
          f"[{obs_exp_pct_rank(row['obs_exp_ratio']):.0f}th percentile]")
    print(f"  LLR:                           {row['llr']:.3f}  "
          f"[{llr_pct_rank(row['llr']):.0f}th percentile]")
    print(f"  Duration:                      {int(row['duration_days'])} days  "
          f"({row['start_date'].strftime('%Y-%m-%d')} → {row['end_date'].strftime('%Y-%m-%d')})")
    print()
    print(f"  ── Plain-English ─────────────────────────────────────────────")

    # Build interpretation from the actual numbers, no thresholds
    offset_desc = (
        "well-centered on the cases"         if offset_pct < 25 else
        "moderately offset from the cases"   if offset_pct < 50 else
        "substantially offset from the cases" if offset_pct < 75 else
        "very far from the cases — sitting near the opposite edge of the circle"
    )
    coverage_desc = (
        "cases fill the circle well"              if coverage > 70 else
        "cases fill a moderate portion of the circle" if coverage > 45 else
        "cases occupy a relatively small portion of the circle area"
    )
    signal_desc = (
        "very strong local signal"   if row["obs_exp_ratio"] > 10 else
        "strong local signal"        if row["obs_exp_ratio"] > 4  else
        "moderate signal"            if row["obs_exp_ratio"] > 2  else
        "weak signal"
    )
    size_desc = (
        f"large ({radius_pct_rank(radius):.0f}th percentile)" 
        if radius_pct_rank(radius) >= 75 
        else f"moderate ({radius_pct_rank(radius):.0f}th percentile)"
    )

    print(
        f"  This is a {size_desc} cluster with a {signal_desc} "
        f"(obs/exp = {row['obs_exp_ratio']:.2f}x, LLR = {row['llr']:.1f}). "
        f"The declared center is {offset_desc} — the case centroid sits "
        f"{offset_pct:.0f}% of the radius away from the declared center "
        f"({offset:.1f} km). "
        f"In terms of spatial fill, {coverage_desc} "
        f"({coverage:.0f}% of circle area covered by the case bounding box). "
    )

    # Offset-specific addendum
    print(
        f"  The declared center is at {offset_pct:.0f}% of the radius from the case centroid "
        f"({offset:.1f} km). The empty portion of the circle "
        f"({'substantial' if coverage < 50 else 'moderate'} at {100-coverage:.0f}% unfilled) "
        f"did not penalize the LLR enough to force repositioning. "
        f"A smaller max spatial window may produce a tighter, better-centered cluster here."
    )

    print()
    print("-" * 65)
    print()

In [ ]:
# =============================================================================
# CELL G — Dataset-wide edge-hugging ranking across all clusters
# =============================================================================

# Compute continuous metrics for every cluster
df["offset_pct_of_radius"] = (
    df["center_to_case_centroid_km"] / df["radius_km"] * 100
)

df["circle_area_km2"] = np.pi * df["radius_km"] ** 2

df["bbox_area_km2"] = (
    df["cases_bbox_lat_span_km"] * df["cases_bbox_lng_span_km"]
)

df["bbox_coverage_pct"] = (
    df["bbox_area_km2"] / df["circle_area_km2"] * 100
).clip(upper=100)

# Percentile ranks within dataset (0–100)
df["radius_pctile"]      = df["radius_km"].rank(pct=True) * 100
df["offset_pctile"]      = df["center_to_case_centroid_km"].rank(pct=True) * 100
df["offset_pct_pctile"]  = df["offset_pct_of_radius"].rank(pct=True) * 100
df["coverage_pctile"]    = df["bbox_coverage_pct"].rank(pct=True) * 100   # high = good fill
df["obs_exp_pctile"]     = df["obs_exp_ratio"].rank(pct=True) * 100
df["llr_pctile"]         = df["llr"].rank(pct=True) * 100

# Composite edge-hugging score:
#   high offset % of radius  → more edge-hugging
#   high radius percentile   → larger circle
#   low coverage %           → cases fill less of the circle
#   low obs/exp              → signal diluted by large area
# All components normalized 0–100, higher = more concerning
df["edge_hugging_score"] = (
    df["offset_pct_pctile"] * 0.40 +    # offset as % of radius (most important)
    df["radius_pctile"]     * 0.30 +    # raw circle size
    (100 - df["coverage_pctile"]) * 0.20 +  # inverse of fill (low fill = bad)
    (100 - df["obs_exp_pctile"]) * 0.10     # inverse of obs/exp (weak signal = bad)
)

# ── Full ranking table ────────────────────────────────────────────────────────
ranking_cols = [
    "cluster_id", "edge_hugging_score",
    "radius_km", "radius_pctile",
    "center_to_case_centroid_km", "offset_pct_of_radius", "offset_pct_pctile",
    "bbox_coverage_pct",
    "obs_exp_ratio", "obs_exp_pctile",
    "llr", "llr_pctile",
    "observed_cases", "duration_days",
    "start_date", "end_date",
]

ranked = (
    df[ranking_cols]
    .sort_values("edge_hugging_score", ascending=False)
    .reset_index(drop=True)
)
ranked.index += 1   # rank starts at 1

print("=== All Clusters Ranked by Edge-Hugging Score (top 30) ===\n")
display(ranked.head(30).round(2))

# ── Highlight where your 4 target clusters land ──────────────────────────────
print("\n=== Where do your 4 target clusters rank? ===\n")
target_ranks = ranked[ranked["cluster_id"].isin(TARGET_IDS)][
    ["cluster_id", "edge_hugging_score",
     "radius_km", "offset_pct_of_radius",
     "bbox_coverage_pct", "obs_exp_ratio", "llr"]
].copy()
target_ranks.index = ranked[ranked["cluster_id"].isin(TARGET_IDS)].index
print(target_ranks.round(2).to_string())

# ── Flag clusters outside your target list worth investigating ────────────────
top30_ids    = set(ranked.head(30)["cluster_id"].tolist())
already_know = set(TARGET_IDS)
new_flags    = top30_ids - already_know

print(f"\n=== Top 30 clusters NOT already in your target list ===")
print(f"    Cluster IDs to consider investigating: {sorted(new_flags)}\n")
display(
    ranked[ranked["cluster_id"].isin(new_flags)]
    [["cluster_id", "edge_hugging_score", "radius_km",
      "offset_pct_of_radius", "bbox_coverage_pct",
      "obs_exp_ratio", "llr", "observed_cases"]]
    .sort_values("edge_hugging_score", ascending=False)
    .reset_index(drop=True)
    .round(2)
)


# ── Visualize the score distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Edge-Hugging Score — Dataset-Wide Distribution",
             fontsize=13, fontweight="bold")

# Left: histogram of scores with target clusters marked
ax = axes[0]
ax.hist(df["edge_hugging_score"], bins=40,
        color="steelblue", alpha=0.8, edgecolor="white")
for cid, color in zip(TARGET_IDS, ["blue","darkorange","green","crimson"]):
    score = df[df["cluster_id"] == cid]["edge_hugging_score"].values[0]
    ax.axvline(score, color=color, linewidth=2,
               linestyle="--", label=f"C{cid} ({score:.1f})")
ax.set_xlabel("Edge-Hugging Score")
ax.set_ylabel("Number of Clusters")
ax.set_title("Score Distribution\n(higher = more edge-hugging concern)")
ax.legend(fontsize=9)

# Right: scatter of offset % vs radius, colored by score, targets labeled
ax = axes[1]
sc = ax.scatter(
    df["radius_km"],
    df["offset_pct_of_radius"],
    c=df["edge_hugging_score"],
    cmap="YlOrRd", s=30, alpha=0.7,
    edgecolors="none"
)
plt.colorbar(sc, ax=ax, label="Edge-Hugging Score")

# Label top 15 and the 4 targets
label_ids = set(ranked.head(15)["cluster_id"].tolist()) | set(TARGET_IDS)
for _, row in df[df["cluster_id"].isin(label_ids)].iterrows():
    color = "black"
    for cid, c in zip(TARGET_IDS, ["blue","darkorange","green","crimson"]):
        if row["cluster_id"] == cid:
            color = c
    ax.annotate(
        f"C{int(row['cluster_id'])}",
        (row["radius_km"], row["offset_pct_of_radius"]),
        fontsize=7.5, color=color, fontweight="bold",
        xytext=(4, 2), textcoords="offset points"
    )

ax.set_xlabel("Cluster Radius (km)")
ax.set_ylabel("Offset as % of Radius")
ax.set_title("Radius vs Offset %\n(upper right = most edge-hugging)")

plt.tight_layout()
plt.show()


# ── Score component breakdown for top 15 ─────────────────────────────────────
top15_ids = ranked.head(15)["cluster_id"].tolist()
top15 = df[df["cluster_id"].isin(top15_ids)].copy()
top15["label"] = top15["cluster_id"].apply(lambda x: f"C{int(x)}")
top15 = top15.sort_values("edge_hugging_score", ascending=True)

components = {
    "Offset % of radius (40%)" : top15["offset_pct_pctile"] * 0.40,
    "Circle size (30%)"         : top15["radius_pctile"]     * 0.30,
    "Inverse fill (20%)"        : (100 - top15["coverage_pctile"]) * 0.20,
    "Inverse obs/exp (10%)"     : (100 - top15["obs_exp_pctile"])  * 0.10,
}

fig, ax = plt.subplots(figsize=(12, 7))
bottom = np.zeros(len(top15))
comp_colors = ["#e74c3c", "#e67e22", "#3498db", "#2ecc71"]

for (label, vals), color in zip(components.items(), comp_colors):
    ax.barh(top15["label"], vals.values, left=bottom,
            color=color, label=label, edgecolor="white", linewidth=0.4)
    bottom += vals.values

ax.set_xlabel("Edge-Hugging Score (component breakdown)")
ax.set_title("Top 15 Clusters — Score Component Breakdown",
             fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=9)

# Mark the 4 known targets
for label_text in ax.get_yticklabels():
    cid_str = label_text.get_text().replace("C", "")
    if cid_str.isdigit() and int(cid_str) in TARGET_IDS:
        label_text.set_color("crimson")
        label_text.set_fontweight("bold")

plt.tight_layout()
plt.show()

# Determining Max Spatial Window Through Gini Coefficient Optimization Simulations

In [ ]:
# =============================================================================
# CELL H1 — Radius Distribution Analysis
# =============================================================================

print("=== Radius Distribution Summary ===\n")
pctiles = [5, 10, 25, 50, 75, 90, 95, 99, 100]
for p in pctiles:
    print(f"  P{p:>3}: {np.percentile(df['radius_km'], p):.4f} km")
print(f"\n  Mean: {df['radius_km'].mean():.4f} km")
print(f"  Std:  {df['radius_km'].std():.4f} km")

print("\n=== Radius Bin Distribution ===\n")
bins  = [0, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0, 15.0, np.inf]
labels = ["0–0.1","0.1–0.25","0.25–0.5","0.5–1.0",
          "1.0–2.0","2.0–5.0","5.0–10.0","10.0–15.0","≥15.0"]
df["radius_bin"] = pd.cut(df["radius_km"], bins=bins, labels=labels)
bin_counts = df["radius_bin"].value_counts().sort_index()
for label, count in bin_counts.items():
    pct = count / len(df) * 100
    bar = "█" * int(pct)
    print(f"  {label:>10} km: {count:>4} ({pct:>5.1f}%)  {bar}")

# ── Plot: radius distribution ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Cluster Radius Distribution", fontsize=13, fontweight="bold")

# Full distribution (log scale)
ax = axes[0]
ax.hist(df["radius_km"], bins=60, color="steelblue", alpha=0.85, edgecolor="white")
ax.set_yscale("log")
ax.set_xlabel("Radius (km)")
ax.set_ylabel("Count (log scale)")
ax.set_title("Full Distribution\n(log y-axis)")

# Zoomed: clusters under 2km (where 98% of data lives)
ax = axes[1]
sub = df[df["radius_km"] < 2]
ax.hist(sub["radius_km"], bins=60, color="darkorange", alpha=0.85, edgecolor="white")
ax.set_xlabel("Radius (km)")
ax.set_ylabel("Count")
ax.set_title(f"Zoomed: clusters < 2 km\n({len(sub)} of {len(df)}, {len(sub)/len(df)*100:.1f}%)")

# Radius vs offset % — where are the large edge-huggers?
ax = axes[2]
sc = ax.scatter(df["radius_km"], df["offset_pct_of_radius"],
                c=df["obs_exp_ratio"], cmap="RdYlGn",
                s=20, alpha=0.6, edgecolors="none")
plt.colorbar(sc, ax=ax, label="Obs/Exp Ratio")
ax.set_xlabel("Radius (km)")
ax.set_ylabel("Offset as % of Radius")
ax.set_title("Radius vs Offset %\n(color = Obs/Exp ratio)")

plt.tight_layout()
plt.show()



In [ ]:

# =============================================================================
# CELL H2 — Define the simulated window sizes
#
# The key question: what is your actual max spatial window in km?
# Your setting is MaxSpatialSizeInPopulationAtRisk = 5%
# This means the largest allowed cluster contains at most 5% of all cases.
# We can back-calculate the effective radius cap from your data.
# =============================================================================

total_cases  = df["observed_cases"].sum()
max_cases_allowed = total_cases * 0.05

print(f"=== Effective Spatial Window Constraints ===\n")
print(f"  Total observed cases in dataset: {total_cases:,.0f}")
print(f"  5% of total cases:               {max_cases_allowed:,.0f}")
print(f"  Largest cluster observed:        {df['radius_km'].max():.2f} km  "
      f"({df['observed_cases'].max():.0f} cases)")
print(f"  90th pctile radius:              {df['radius_km'].quantile(0.90):.4f} km")
print(f"  95th pctile radius:              {df['radius_km'].quantile(0.95):.4f} km")
print(f"  99th pctile radius:              {df['radius_km'].quantile(0.99):.4f} km")

# Simulate tighter windows by radius caps derived from percentiles of your data
# This is the key simulation: "what would my results look like if SaTScan
# had been constrained to a tighter max spatial window?"

# Define simulated window caps (in km) based on your data's own percentiles
window_caps = {
    "Current (~5% pop)"    : df["radius_km"].max(),           # no cap = current run
    "P99 cap (2.5 km)"     : df["radius_km"].quantile(0.99),
    "P95 cap (0.94 km)"    : df["radius_km"].quantile(0.95),
    "P90 cap (0.46 km)"    : df["radius_km"].quantile(0.90),
    "P75 cap (0.21 km)"    : df["radius_km"].quantile(0.75),
}

print(f"\n=== Simulated Window Caps ===\n")
for label, cap in window_caps.items():
    n_kept = (df["radius_km"] <= cap).sum()
    pct_kept = n_kept / len(df) * 100
    print(f"  {label:<25}: cap = {cap:.3f} km  →  "
          f"{n_kept} clusters retained ({pct_kept:.1f}%)")



In [ ]:

# =============================================================================
# CELL H3 — Gini coefficient computation and optimization
# =============================================================================

def gini_coefficient(values: np.ndarray) -> float:
    """
    Standard Gini coefficient of an array of non-negative values.
    0 = perfect equality (all clusters same intensity)
    1 = perfect inequality (one cluster has all the signal)
    Higher = more concentrated signal = better-defined clusters.
    """
    v = np.sort(np.abs(values))
    n = len(v)
    if n == 0 or v.sum() == 0:
        return 0.0
    cumv = np.cumsum(v)
    return (2 * np.sum(np.arange(1, n + 1) * v) / (n * cumv[-1])) - (n + 1) / n


def compute_window_metrics(df_sub: pd.DataFrame, label: str) -> dict:
    """Compute a full set of metrics for a given filtered cluster set."""
    if len(df_sub) == 0:
        return {}

    offset_pct = df_sub["center_to_case_centroid_km"] / df_sub["radius_km"] * 100
    circle_area = np.pi * df_sub["radius_km"] ** 2
    bbox_area   = df_sub["cases_bbox_lat_span_km"] * df_sub["cases_bbox_lng_span_km"]
    coverage    = (bbox_area / circle_area * 100).clip(upper=100)

    return {
        "label"                  : label,
        "n_clusters"             : len(df_sub),
        # Gini on different signals — higher is better (more concentrated)
        "gini_obs_exp"           : gini_coefficient(df_sub["obs_exp_ratio"].values),
        "gini_llr"               : gini_coefficient(df_sub["llr"].values),
        "gini_excess_cases"      : gini_coefficient(df_sub["excess_cases"].values),
        # Radius metrics
        "mean_radius_km"         : df_sub["radius_km"].mean(),
        "median_radius_km"       : df_sub["radius_km"].median(),
        "max_radius_km"          : df_sub["radius_km"].max(),
        # Edge-hugging metrics — lower is better
        "mean_offset_pct"        : offset_pct.mean(),
        "median_offset_pct"      : offset_pct.median(),
        "pct_offset_gt50"        : (offset_pct > 50).mean() * 100,
        # Fill metrics — higher is better
        "mean_coverage_pct"      : coverage.mean(),
        "median_coverage_pct"    : coverage.median(),
        # Signal strength
        "mean_obs_exp"           : df_sub["obs_exp_ratio"].mean(),
        "median_obs_exp"         : df_sub["obs_exp_ratio"].median(),
        "mean_llr"               : df_sub["llr"].mean(),
        "total_excess_cases"     : df_sub["excess_cases"].sum(),
    }


# Run metrics across all simulated window caps
results = []
for label, cap in window_caps.items():
    df_sub = df[df["radius_km"] <= cap].copy()
    metrics = compute_window_metrics(df_sub, label)
    results.append(metrics)

metrics_df = pd.DataFrame(results).set_index("label")

print("=== Gini Optimization Results Across Simulated Windows ===\n")
display_cols = [
    "n_clusters", "gini_obs_exp", "gini_llr", "gini_excess_cases",
    "mean_radius_km", "mean_offset_pct", "pct_offset_gt50",
    "mean_coverage_pct", "mean_obs_exp", "mean_llr"
]
display(metrics_df[display_cols].round(4))

# Identify optimal window
best_gini_obs_exp = metrics_df["gini_obs_exp"].idxmax()
best_gini_llr     = metrics_df["gini_llr"].idxmax()
print(f"\n  Best window by Gini(obs/exp): {best_gini_obs_exp}")
print(f"  Best window by Gini(LLR):     {best_gini_llr}")



In [ ]:

# =============================================================================
# CELL H4 — Visualize the Gini optimization
# =============================================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Simulated Spatial Window Optimization — Gini Coefficient Analysis",
             fontsize=13, fontweight="bold")

window_labels = list(metrics_df.index)
x = range(len(window_labels))
bar_color = "steelblue"
highlight = "crimson"

def bar_plot(ax, values, title, ylabel, higher_better=True):
    colors = [highlight if v == (max(values) if higher_better else min(values))
              else bar_color for v in values]
    bars = ax.bar(x, values, color=colors, alpha=0.85, edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels(window_labels, rotation=30, ha="right", fontsize=8.5)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel(ylabel)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)
    best_label = "▲ best" if higher_better else "▼ best"
    ax.annotate(best_label,
                xy=(colors.index(highlight), max(values) if higher_better else min(values)),
                xytext=(0, 12), textcoords="offset points",
                ha="center", fontsize=8, color=highlight, fontweight="bold")

bar_plot(axes[0,0], metrics_df["gini_obs_exp"].values,
         "Gini(Obs/Exp Ratio)\n↑ higher = more concentrated signal",
         "Gini Coefficient", higher_better=True)

bar_plot(axes[0,1], metrics_df["gini_llr"].values,
         "Gini(LLR)\n↑ higher = more concentrated evidence",
         "Gini Coefficient", higher_better=True)

bar_plot(axes[0,2], metrics_df["mean_offset_pct"].values,
         "Mean Offset % of Radius\n↓ lower = less edge-hugging",
         "Mean Offset (%)", higher_better=False)

bar_plot(axes[1,0], metrics_df["mean_coverage_pct"].values,
         "Mean Case Bbox Coverage %\n↑ higher = cases fill circle better",
         "Coverage (%)", higher_better=True)

bar_plot(axes[1,1], metrics_df["mean_obs_exp"].values,
         "Mean Obs/Exp Ratio\n↑ higher = stronger average signal",
         "Obs/Exp Ratio", higher_better=True)

bar_plot(axes[1,2], metrics_df["n_clusters"].values,
         "Clusters Retained\n(tradeoff: tighter window = fewer clusters)",
         "N Clusters", higher_better=False)

plt.tight_layout()
plt.show()



In [ ]:

# =============================================================================
# CELL H5 — Tradeoff curve: Gini vs clusters lost
# =============================================================================

# Finer-grained simulation across many radius caps
caps_km = np.percentile(df["radius_km"],
                        np.arange(50, 101, 1))  # P50 to P100 in 1% steps
caps_km = np.unique(caps_km)

fine_results = []
for cap in caps_km:
    df_sub = df[df["radius_km"] <= cap]
    if len(df_sub) < 10:
        continue
    offset_pct = df_sub["center_to_case_centroid_km"] / df_sub["radius_km"] * 100
    fine_results.append({
        "cap_km"          : cap,
        "n_clusters"      : len(df_sub),
        "pct_retained"    : len(df_sub) / len(df) * 100,
        "gini_obs_exp"    : gini_coefficient(df_sub["obs_exp_ratio"].values),
        "gini_llr"        : gini_coefficient(df_sub["llr"].values),
        "mean_offset_pct" : offset_pct.mean(),
        "mean_obs_exp"    : df_sub["obs_exp_ratio"].mean(),
    })

fine_df = pd.DataFrame(fine_results)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Continuous Tradeoff Curves — Simulated Spatial Window Tightening",
             fontsize=13, fontweight="bold")

ax = axes[0, 0]
ax.plot(fine_df["cap_km"], fine_df["gini_obs_exp"],
        color="steelblue", linewidth=2)
ax.set_xlabel("Max Radius Cap (km)")
ax.set_ylabel("Gini(Obs/Exp)")
ax.set_title("Gini Concentration vs Radius Cap\n(↑ higher is better)")
best_idx = fine_df["gini_obs_exp"].idxmax()
ax.axvline(fine_df.loc[best_idx, "cap_km"], color="crimson",
           linestyle="--", linewidth=1.5,
           label=f"Best: {fine_df.loc[best_idx, 'cap_km']:.3f} km")
ax.legend()

ax = axes[0, 1]
ax.plot(fine_df["cap_km"], fine_df["mean_offset_pct"],
        color="darkorange", linewidth=2)
ax.set_xlabel("Max Radius Cap (km)")
ax.set_ylabel("Mean Offset % of Radius")
ax.set_title("Edge-Hugging vs Radius Cap\n(↓ lower is better)")

ax = axes[1, 0]
ax.plot(fine_df["cap_km"], fine_df["pct_retained"],
        color="mediumseagreen", linewidth=2)
ax.set_xlabel("Max Radius Cap (km)")
ax.set_ylabel("% of Clusters Retained")
ax.set_title("Clusters Retained vs Radius Cap\n(tradeoff: tighter = fewer clusters)")
ax.axhline(90, color="gray", linestyle=":", linewidth=1, label="90% retained")
ax.axhline(75, color="gray", linestyle="--", linewidth=1, label="75% retained")
ax.legend()

ax = axes[1, 1]
ax2 = ax.twinx()
ax.plot(fine_df["cap_km"], fine_df["gini_obs_exp"],
        color="steelblue", linewidth=2, label="Gini(Obs/Exp)")
ax2.plot(fine_df["cap_km"], fine_df["pct_retained"],
         color="mediumseagreen", linewidth=2, linestyle="--",
         label="% clusters retained")
ax.set_xlabel("Max Radius Cap (km)")
ax.set_ylabel("Gini Coefficient", color="steelblue")
ax2.set_ylabel("% Clusters Retained", color="mediumseagreen")
ax.set_title("Gini vs Clusters Retained\n(find the elbow)")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

plt.tight_layout()
plt.show()



In [ ]:

# =============================================================================
# CELL H6 — Summary interpretation
# =============================================================================

print("=== Simulation Summary & Interpretation ===\n")

best_gini_cap  = fine_df.loc[fine_df["gini_obs_exp"].idxmax(), "cap_km"]
best_gini_val  = fine_df["gini_obs_exp"].max()
current_gini   = fine_df.iloc[-1]["gini_obs_exp"]   # largest cap = current run
pct_at_best    = fine_df.loc[fine_df["gini_obs_exp"].idxmax(), "pct_retained"]

print(f"  Current run (max radius {df['radius_km'].max():.2f} km):")
print(f"    Gini(Obs/Exp):      {current_gini:.4f}")
print(f"    Mean offset %:      {(df['center_to_case_centroid_km']/df['radius_km']*100).mean():.1f}%")
print()
print(f"  Simulated optimal window ({best_gini_cap:.3f} km cap):")
print(f"    Gini(Obs/Exp):      {best_gini_val:.4f}  "
      f"(+{(best_gini_val-current_gini)/current_gini*100:.1f}% vs current)")
print(f"    Clusters retained:  {pct_at_best:.1f}% of current clusters")
print()
print("  Interpretation:")
print(f"  The simulation suggests that capping the maximum cluster radius at")
print(f"  ~{best_gini_cap:.2f} km would maximize signal concentration (Gini = {best_gini_val:.4f}),")
print(f"  retaining {pct_at_best:.1f}% of clusters. This is consistent with the")
print(f"  observed edge-hugging in large clusters — the {100-pct_at_best:.1f}% of clusters")
print(f"  excluded at this threshold are the large, diffuse, low-obs/exp clusters")
print(f"  that are diluting the overall Gini score.")
print()
print(f"  NOTE: This is a post-hoc simulation, not a true re-run of SaTScan.")
print(f"  The true optimal MaxSpatialSizeInPopulationAtRisk would require")
print(f"  re-running SaTScan with different settings. This simulation provides")
print(f"  directional evidence for what that parameter change would likely produce.")